<a href="https://www.kaggle.com/code/avikdas567/predict-1-year-us-stock-returns-from-fundamentals?scriptVersionId=311242850" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

import lightgbm as lgb
import catboost as cb

SEED = 42
N_SPLITS = 5
TARGET = "return_pct"

# =========================
# LOAD DATA
# =========================
train = pd.read_csv("/kaggle/input/competitions/predict-1-year-us-stock-returns-from-fundamentals/train.csv")
test = pd.read_csv("/kaggle/input/competitions/predict-1-year-us-stock-returns-from-fundamentals/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/predict-1-year-us-stock-returns-from-fundamentals/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# =========================
# SAFE PREPROCESS
# =========================
def preprocess(df):
    df = df.copy()
    
    if "period_start" in df.columns:
        df["period_start"] = pd.to_datetime(df["period_start"])
        df["year"] = df["period_start"].dt.year
        df["month"] = df["period_start"].dt.month
    else:
        df["year"] = 2024
        df["month"] = 1
    
    if "period_end" in df.columns:
        df["period_end"] = pd.to_datetime(df["period_end"])
    
    return df

train = preprocess(train)
test = preprocess(test)

# =========================
# FEATURE ENGINEERING
# =========================
def feature_engineering(df):
    df = df.copy()
    
    log_cols = [
        "revenue_ttm", "net_income_ttm", "total_assets",
        "stockholders_equity", "current_assets", "long_term_debt"
    ]
    for col in log_cols:
        if col in df.columns:
            df[f"log_{col}"] = np.log1p(np.abs(df[col]))
    
    df["profit_to_assets"] = df["net_income_ttm"] / (df["total_assets"] + 1e-6)
    df["income_to_equity"] = df["net_income_ttm"] / (df["stockholders_equity"] + 1e-6)
    
    df["debt_ratio"] = df["long_term_debt"] / (df["total_assets"] + 1e-6)
    df["asset_turnover"] = df["revenue_ttm"] / (df["total_assets"] + 1e-6)
    
    df["valuation_efficiency"] = df["roe"] / (df["pe_ttm"] + 1e-6)
    df["growth_adjusted_pe"] = df["pe_ttm"] / (df["revenue_growth_yoy"] + 1e-6)
    
    df["liquidity_spread"] = df["current_ratio"] - df["quick_ratio"]
    
    return df

train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# SECTOR FEATURES
# =========================
def add_sector_features(df):
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in num_cols:
        if col not in ["id", TARGET]:
            grp = df.groupby("sector_code")[col]
            df[f"{col}_sector_mean"] = grp.transform("mean")
            df[f"{col}_sector_std"] = grp.transform("std")
            df[f"{col}_sector_z"] = (df[col] - df[f"{col}_sector_mean"]) / (df[f"{col}_sector_std"] + 1e-6)
    
    return df

train = add_sector_features(train)
test = add_sector_features(test)

# =========================
# TARGET CLIP
# =========================
train[TARGET] = np.clip(train[TARGET], -100, 200)

# =========================
# FEATURES
# =========================
drop_cols = ["id", "ticker", "period_start", "period_end", TARGET]
features = [c for c in train.columns if c not in drop_cols]

# =========================
# MISSING VALUES
# =========================
for col in features:
    median = train[col].median()
    train[col] = train[col].fillna(median)
    test[col] = test[col].fillna(median)

# =========================
# SCALING
# =========================
scaler = RobustScaler()
train_scaled = scaler.fit_transform(train[features])
test_scaled = scaler.transform(test[features])

# =========================
# CV
# =========================
groups = train["ticker"]
gkf = GroupKFold(n_splits=N_SPLITS)

oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
oof_ridge = np.zeros(len(train))

pred_lgb = np.zeros(len(test))
pred_cat = np.zeros(len(test))
pred_ridge = np.zeros(len(test))

# =========================
# TRAIN LOOP
# =========================
for fold, (tr_idx, val_idx) in enumerate(gkf.split(train, groups=groups)):
    print(f"\n===== FOLD {fold+1} =====")
    
    X_tr, X_val = train.iloc[tr_idx][features], train.iloc[val_idx][features]
    y_tr, y_val = train.iloc[tr_idx][TARGET], train.iloc[val_idx][TARGET]
    
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        device="gpu"
    )
    
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    pred_lgb += lgb_model.predict(test[features]) / N_SPLITS
    
    cat_model = cb.CatBoostRegressor(
        iterations=800,
        learning_rate=0.02,
        depth=6,
        loss_function="RMSE",
        verbose=0,
        random_seed=SEED
    )
    
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100, verbose=0)
    
    oof_cat[val_idx] = cat_model.predict(X_val)
    pred_cat += cat_model.predict(test[features]) / N_SPLITS
    
    ridge = Ridge(alpha=10)
    ridge.fit(train_scaled[tr_idx], y_tr)
    
    oof_ridge[val_idx] = ridge.predict(train_scaled[val_idx])
    pred_ridge += ridge.predict(test_scaled) / N_SPLITS

# =========================
# ENSEMBLE
# =========================
oof = 0.5 * oof_lgb + 0.3 * oof_cat + 0.2 * oof_ridge
pred = 0.5 * pred_lgb + 0.3 * pred_cat + 0.2 * pred_ridge

rmse = np.sqrt(mean_squared_error(train[TARGET], oof))
print("\nCV RMSE:", rmse)

# =========================
# SUBMISSION
# =========================
pred = np.clip(pred, -100, 200)

submission = sample.copy()
submission["return_pct"] = pred
submission.to_csv("submission.csv", index=False)

print("\n'submission.csv' created.")

Train shape: (23070, 39)
Test shape: (8520, 36)

===== FOLD 1 =====
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 24249
[LightGBM] [Info] Number of data points in the train set: 18456, number of used features: 194
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 188 dense feature groups (3.31 MB) transferred to GPU in 0.003775 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 12.624081

===== FOLD 2 =====
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 24214
[LightGBM] [Info] Number of data points in the train set: 18456, number of used features: 194
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 188 dense feature groups (3.31 MB) transferred to GPU in 0.003676 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 13.795230

===== FOLD 3 =====
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 24309
[LightGBM] [Info] Number of data points in the t